# DEPRECATED — pipeline SAM 3 đã được thay bằng SAM Base

Không chạy notebook này cho thí nghiệm hiện tại. Dùng `TRAIN_SAM_BASE_SEGMENT_ROBOFLOW.ipynb`, nạp `MODELS/sam_b.pt`. File này chỉ được giữ tạm vì đang bị workspace khóa nên chưa thể xóa an toàn.

Trước khi chạy:
1. Chọn GPU runtime trong Colab. A100 40 GB được khuyến nghị cho profile `quality`; T4/L4 có thể thiếu VRAM.
2. Trong Colab Secrets tạo `ROBOFLOW_API_KEY` và `HF_TOKEN`. Không ghi token trực tiếp vào notebook.
3. Xin quyền truy cập checkpoint `facebook/sam3` trên Hugging Face trước khi chạy.
4. Kết quả SAM 3 không thay thế YOLO tự động; phải so sánh mask AP, count error, tốc độ và VRAM trước.

> Lưu ý mật độ: SAM 3 chính thức mặc định dùng 200 object queries. Notebook thống kê số instance/ảnh và cảnh báo nếu ảnh có hơn 200 hạt, vì đó có thể là giới hạn quan trọng đối với bài toán đếm.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Cùng nguồn dữ liệu với pipeline YOLO để phép so sánh có ý nghĩa.
ROBOFLOW_WORKSPACE = 'minh-tr-tklp0'
ROBOFLOW_PROJECT = 'full-grains-segmentaion'
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'coco-segmentation'

# quality: 1008 px, ưu tiên A100 >= 40 GB.
# economy: 504 px, chỉ dùng thử khi GPU ít VRAM; không dùng số liệu này để kết luận thay quality.
PROFILE = 'quality'
PROFILES = {
    'quality': {'resolution': 1008, 'min_resize': 480, 'grad_accum': 1},
    'economy': {'resolution': 504, 'min_resize': 238, 'grad_accum': 4},
}
EPOCHS = 20
SEED = 123
VAL_EPOCH_FREQ = 5
MAX_EXPECTED_INSTANCES = 200
STRICT_DENSITY_CHECK = False

SAM3_ROOT = Path('/content/sam3')
REQUESTED_DATASET_DIR = Path('/content/datasets/full-grains-segmentaion-v1')
RUN_ROOT = Path('/content/sam3_runs/rice_grains_sam3')
DRIVE_EXPORT_ROOT = Path('/content/drive/MyDrive/Research-RICAI/SAM3/rice_grains_v1')

assert PROFILE in PROFILES
print('Profile:', PROFILE, PROFILES[PROFILE])
print('Local run:', RUN_ROOT)
print('Drive export:', DRIVE_EXPORT_ROOT)

## 1. Cài SAM 3 chính thức và dependencies

Notebook không tự hạ/cài lại PyTorch để tránh làm sai CUDA runtime của Colab. Cell preflight kế tiếp sẽ dừng sớm nếu Python, PyTorch hoặc CUDA không đạt yêu cầu của SAM 3 hiện hành. Sau khi cài package, nếu Colab yêu cầu restart runtime thì restart và chạy lại từ đầu.

In [ ]:
import os
import subprocess

if not SAM3_ROOT.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/facebookresearch/sam3.git',
        str(SAM3_ROOT),
    ], check=True)

subprocess.run([
    'python', '-m', 'pip', 'install', '-q',
    '-e', f'{SAM3_ROOT}[train]',
    'roboflow', 'huggingface_hub', 'pycocotools',
], check=True)

commit = subprocess.check_output(
    ['git', '-C', str(SAM3_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
print('SAM 3 commit:', commit)

In [ ]:
import sys
import torch
from packaging.version import Version
from google.colab import userdata
from huggingface_hub import HfApi, login

assert sys.version_info >= (3, 12), f'SAM 3 cần Python >= 3.12, hiện tại: {sys.version}'
torch_version = Version(torch.__version__.split('+')[0])
assert torch_version >= Version('2.7'), f'SAM 3 cần PyTorch >= 2.7, hiện tại: {torch.__version__}'
assert torch.cuda.is_available(), 'Chưa có CUDA GPU. Hãy chọn GPU runtime trong Colab.'

cuda_version = Version(torch.version.cuda)
assert cuda_version >= Version('12.6'), f'SAM 3 cần CUDA runtime >= 12.6, hiện tại: {torch.version.cuda}'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
capability = torch.cuda.get_device_capability(0)
AMP_DTYPE = 'bfloat16' if capability[0] >= 8 else 'float16'
print(f'GPU: {gpu_name} | VRAM: {gpu_mem_gb:.1f} GB | capability: {capability}')
print(f'Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
print('AMP dtype:', AMP_DTYPE)
if PROFILE == 'quality' and gpu_mem_gb < 35:
    print('WARNING: profile quality có thể OOM trên GPU dưới 35 GB. Dùng economy chỉ để thử kỹ thuật.')

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN')
assert ROBOFLOW_API_KEY, 'Thiếu Colab Secret ROBOFLOW_API_KEY.'
assert HF_TOKEN, 'Thiếu Colab Secret HF_TOKEN.'
login(token=HF_TOKEN, add_to_git_credential=False)
model_info = HfApi(token=HF_TOKEN).model_info('facebook/sam3')
print('Hugging Face access OK:', model_info.id)

## 2. Tải đúng Roboflow version dưới dạng COCO segmentation

Không dùng export `yolov8` cho SAM 3. COCO JSON giữ polygon/RLE mask và category name dùng làm text prompt.

In [ ]:
from roboflow import Roboflow

annotation_probe = REQUESTED_DATASET_DIR / 'train' / '_annotations.coco.json'
if annotation_probe.exists():
    DATASET_DIR = REQUESTED_DATASET_DIR.resolve()
    print('Reuse dataset:', DATASET_DIR)
else:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download(ROBOFLOW_FORMAT, location=str(REQUESTED_DATASET_DIR))
    DATASET_DIR = Path(dataset.location).resolve()

print('Dataset directory:', DATASET_DIR)
for split in ('train', 'valid', 'test'):
    print(split, '->', DATASET_DIR / split / '_annotations.coco.json')

In [ ]:
import json
import statistics
from collections import Counter

def inspect_coco(split):
    split_dir = DATASET_DIR / split
    ann_path = split_dir / '_annotations.coco.json'
    assert ann_path.exists(), f'Thiếu annotation: {ann_path}'
    data = json.loads(ann_path.read_text(encoding='utf-8'))
    assert data.get('images'), f'{split}: không có image record'
    assert data.get('annotations'), f'{split}: không có annotation'
    missing_images = [x['file_name'] for x in data['images'] if not (split_dir / x['file_name']).exists()]
    assert not missing_images, f'{split}: thiếu {len(missing_images)} image file, ví dụ {missing_images[:3]}'
    missing_masks = [a.get('id') for a in data['annotations'] if not a.get('segmentation')]
    assert not missing_masks, f'{split}: có {len(missing_masks)} annotation không có segmentation'
    counts = Counter(a['image_id'] for a in data['annotations'])
    per_image = [counts.get(x['id'], 0) for x in data['images']]
    categories = {c['id']: c['name'] for c in data.get('categories', [])}
    summary = {
        'images': len(data['images']),
        'instances': len(data['annotations']),
        'categories': categories,
        'mean_instances': statistics.mean(per_image),
        'max_instances': max(per_image),
        'images_over_200': sum(n > MAX_EXPECTED_INSTANCES for n in per_image),
    }
    return summary

DATASET_SUMMARY = {split: inspect_coco(split) for split in ('train', 'valid', 'test')}
print(json.dumps(DATASET_SUMMARY, ensure_ascii=False, indent=2))

over_limit = sum(x['images_over_200'] for x in DATASET_SUMMARY.values())
if over_limit:
    message = (
        f'Có {over_limit} ảnh vượt {MAX_EXPECTED_INSTANCES} instance. '
        'SAM 3 mặc định có thể bị giới hạn số dự đoán; count comparison với YOLO phải ghi chú điều này.'
    )
    if STRICT_DENSITY_CHECK:
        raise RuntimeError(message)
    print('WARNING:', message)

## 3. Sinh Hydra config riêng cho hạt lúa

Cell này bắt đầu từ config Roboflow chính thức của SAM 3 rồi bật segmentation head, mask loss, COCO mask evaluator, lưu checkpoint, dùng `valid` trong lúc train và giữ `test` cho đánh giá cuối. Config sinh ra nằm trong clone tạm ở `/content`; không sửa source project.

In [ ]:
from omegaconf import OmegaConf

BASE_CONFIG = SAM3_ROOT / 'sam3/train/configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml'
CONFIG_REL = 'configs/rice_grains/rice_grains_sam3.yaml'
CONFIG_PATH = SAM3_ROOT / 'sam3/train' / CONFIG_REL
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

cfg = OmegaConf.load(BASE_CONFIG)
cfg.paths.roboflow_vl_100_root = str(DATASET_DIR.parent)
cfg.paths.experiment_log_dir = str(RUN_ROOT / 'train')
cfg.paths.bpe_path = str(SAM3_ROOT / 'assets/bpe_simple_vocab_16e6.txt.gz')
cfg.roboflow_train.supercategory = DATASET_DIR.name
cfg.roboflow_train.num_images = None

profile = PROFILES[PROFILE]
cfg.scratch.enable_segmentation = True
cfg.scratch.resolution = profile['resolution']
cfg.roboflow_train.train_transforms[0].transforms[3].sizes.min_size = profile['min_resize']
cfg.scratch.gradient_accumulation_steps = profile['grad_accum']
cfg.scratch.train_batch_size = 1
cfg.scratch.val_batch_size = 1
cfg.scratch.num_train_workers = 2
cfg.scratch.num_val_workers = 1
cfg.scratch.max_ann_per_img = 500

# Mask loss: official Roboflow template contains box/class loss only by default.
mask_loss = {
    '_target_': 'sam3.train.loss.loss_fns.Masks',
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
    'weight_dict': {'loss_mask': 200.0, 'loss_dice': 10.0},
    'compute_aux': False,
}
cfg.roboflow_train.loss.loss_fns_find.append(OmegaConf.create(mask_loss))
cfg.roboflow_train.loss.loss_fn_semantic_seg = None

cfg.trainer.skip_saving_ckpts = False
cfg.trainer.skip_first_val = True
cfg.trainer.max_epochs = EPOCHS
cfg.trainer.seed_value = SEED
cfg.trainer.val_epoch_freq = VAL_EPOCH_FREQ
cfg.trainer.mode = 'train'
cfg.trainer.model.enable_segmentation = True
cfg.trainer.model.load_from_HF = True
cfg.trainer.model.checkpoint_path = None
cfg.trainer.optim.amp.amp_dtype = AMP_DTYPE
cfg.trainer.checkpoint.save_dir = str(RUN_ROOT / 'train/checkpoints')
cfg.trainer.checkpoint.save_freq = 0

# Trong train chỉ đánh giá trên valid; test được dành cho cell final evaluation.
valid_ann = DATASET_DIR / 'valid/_annotations.coco.json'
cfg.trainer.data.val.dataset.img_folder = str(DATASET_DIR / 'valid') + '/'
cfg.trainer.data.val.dataset.ann_file = str(valid_ann)

mask_postprocessor = {
    '_target_': 'sam3.eval.postprocessors.PostProcessImage',
    'max_dets_per_img': -1,
    'iou_type': 'segm',
    'use_original_ids': True,
    'use_original_sizes_box': True,
    'use_original_sizes_mask': True,
    'convert_mask_to_rle': True,
    'use_presence': '${scratch.use_presence_eval}',
}
cfg.scratch.mask_postprocessor = OmegaConf.create(mask_postprocessor)

# Giữ bbox AP và thêm mask AP; maxdets cao hơn 100 vì ảnh hạt có mật độ lớn.
det_meter = cfg.trainer.meters.val.roboflow100.detection
det_meter.maxdets = 1000
det_meter.pred_file_evaluators[0].gt_path = str(valid_ann)
seg_meter = {
    '_target_': 'sam3.eval.coco_writer.PredictionDumper',
    'iou_type': 'segm',
    'dump_dir': str(RUN_ROOT / 'train/dumps/valid_masks'),
    'merge_predictions': True,
    'postprocessor': '${scratch.mask_postprocessor}',
    'gather_pred_via_filesys': '${scratch.gather_pred_via_filesys}',
    'maxdets': 1000,
    'pred_file_evaluators': [{
        '_target_': 'sam3.eval.coco_eval_offline.CocoEvaluatorOfflineWithPredFileEvaluators',
        'gt_path': str(valid_ann),
        'tide': False,
        'iou_type': 'segm',
    }],
}
cfg.trainer.meters.val.roboflow100.segmentation = OmegaConf.create(seg_meter)

cfg.launcher.num_nodes = 1
cfg.launcher.gpus_per_node = 1
cfg.launcher.experiment_log_dir = str(RUN_ROOT / 'train')
cfg.submitit.use_cluster = False
cfg.submitit.cpus_per_task = 4
cfg.submitit.job_array.num_tasks = 0

OmegaConf.save(cfg, CONFIG_PATH, resolve=False)
print('Wrote:', CONFIG_PATH)
print(OmegaConf.to_yaml(cfg, resolve=False)[:5000])

In [ ]:
# Preflight config: resolve toàn bộ interpolation trước khi chiếm GPU nhiều giờ.
from hydra import compose, initialize_config_module
from sam3.train.utils.train_utils import register_omegaconf_resolvers

register_omegaconf_resolvers()
with initialize_config_module(config_module='sam3.train', version_base='1.2'):
    composed = compose(config_name=CONFIG_REL)
    resolved = OmegaConf.to_container(composed, resolve=True)

assert resolved['scratch']['enable_segmentation'] is True
assert resolved['trainer']['model']['enable_segmentation'] is True
assert resolved['trainer']['data']['val']['dataset']['ann_file'] == str(valid_ann)
assert resolved['trainer']['meters']['val']['roboflow100']['segmentation']['iou_type'] == 'segm'
print('Hydra config preflight: PASS')

## 4. Train SAM 3

Lần đầu model sẽ tải checkpoint từ Hugging Face. `checkpoint.pt` là checkpoint trainer (chứa model, optimizer, epoch và loss state), không phải file weight thuần như YOLO `best.pt`. Nếu OOM: restart runtime, chọn `PROFILE='economy'`, chạy lại từ đầu và ghi rõ đây là thí nghiệm độ phân giải thấp.

In [ ]:
import subprocess
import torch

torch.cuda.empty_cache()
train_cmd = [
    'python', 'sam3/train/train.py',
    '-c', CONFIG_REL,
    '--use-cluster', '0',
    '--num-gpus', '1',
    '--num-nodes', '1',
]
print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=SAM3_ROOT, check=True)

In [ ]:
# Xác nhận checkpoint thật sự được sinh ra; không coi train là PASS nếu file thiếu/rỗng.
import json
import torch

CHECKPOINT_PATH = RUN_ROOT / 'train/checkpoints/checkpoint.pt'
assert CHECKPOINT_PATH.exists(), f'Không tìm thấy checkpoint: {CHECKPOINT_PATH}'
assert CHECKPOINT_PATH.stat().st_size > 0, 'Checkpoint rỗng.'
checkpoint_meta = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
print({
    'checkpoint': str(CHECKPOINT_PATH),
    'size_gb': round(CHECKPOINT_PATH.stat().st_size / 1024**3, 3),
    'epoch': checkpoint_meta.get('epoch'),
    'steps': checkpoint_meta.get('steps'),
    'keys': sorted(checkpoint_meta.keys()),
})
del checkpoint_meta

## 5. Đánh giá cuối trên test split

Cell này tạo config evaluation riêng, nạp chính checkpoint trainer vừa huấn luyện và đo cả bbox AP lẫn mask AP trên `test`. Không dùng test để chọn epoch/hyperparameter.

In [ ]:
EVAL_CONFIG_REL = 'configs/rice_grains/rice_grains_sam3_test.yaml'
EVAL_CONFIG_PATH = SAM3_ROOT / 'sam3/train' / EVAL_CONFIG_REL
eval_cfg = OmegaConf.load(CONFIG_PATH)
test_run = RUN_ROOT / 'test_eval'
test_ann = DATASET_DIR / 'test/_annotations.coco.json'

eval_cfg.paths.experiment_log_dir = str(test_run)
eval_cfg.trainer.mode = 'val'
eval_cfg.trainer.skip_saving_ckpts = True
eval_cfg.trainer.data.train = None
eval_cfg.trainer.data.val.dataset.img_folder = str(DATASET_DIR / 'test') + '/'
eval_cfg.trainer.data.val.dataset.ann_file = str(test_ann)
eval_cfg.trainer.checkpoint.save_dir = str(test_run / 'checkpoints')
eval_cfg.trainer.checkpoint.resume_from = str(CHECKPOINT_PATH)
eval_cfg.trainer.meters.val.roboflow100.detection.dump_dir = str(test_run / 'dumps/test_boxes')
eval_cfg.trainer.meters.val.roboflow100.detection.pred_file_evaluators[0].gt_path = str(test_ann)
eval_cfg.trainer.meters.val.roboflow100.segmentation.dump_dir = str(test_run / 'dumps/test_masks')
eval_cfg.trainer.meters.val.roboflow100.segmentation.pred_file_evaluators[0].gt_path = str(test_ann)
eval_cfg.launcher.experiment_log_dir = str(test_run)
OmegaConf.save(eval_cfg, EVAL_CONFIG_PATH, resolve=False)

eval_cmd = [
    'python', 'sam3/train/train.py',
    '-c', EVAL_CONFIG_REL,
    '--use-cluster', '0',
    '--num-gpus', '1',
    '--num-nodes', '1',
]
print('Running:', ' '.join(eval_cmd))
subprocess.run(eval_cmd, cwd=SAM3_ROOT, check=True)

In [ ]:
# Lưu config, metrics, dumps và checkpoint sang Drive sau khi train/eval hoàn tất.
import json
import shutil
from datetime import datetime, timezone

DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'sam3_git_commit': commit,
    'roboflow_workspace': ROBOFLOW_WORKSPACE,
    'roboflow_project': ROBOFLOW_PROJECT,
    'roboflow_version': ROBOFLOW_VERSION,
    'roboflow_format': ROBOFLOW_FORMAT,
    'profile': PROFILE,
    'epochs': EPOCHS,
    'seed': SEED,
    'amp_dtype': AMP_DTYPE,
    'gpu': gpu_name,
    'dataset_summary': DATASET_SUMMARY,
}
(RUN_ROOT / 'experiment_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

destination = DRIVE_EXPORT_ROOT / f'{PROFILE}_seed_{SEED}'
if destination.exists():
    destination = DRIVE_EXPORT_ROOT / f'{PROFILE}_seed_{SEED}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
shutil.copytree(RUN_ROOT, destination)
shutil.copy2(CONFIG_PATH, destination / 'rice_grains_sam3.yaml')
shutil.copy2(EVAL_CONFIG_PATH, destination / 'rice_grains_sam3_test.yaml')
print('Export complete:', destination)

## 6. Theo dõi và quy tắc so sánh với YOLO

Có thể mở TensorBoard bằng:

```python
%load_ext tensorboard
%tensorboard --logdir /content/sam3_runs/rice_grains_sam3/train/tensorboard
```

Khi so sánh, dùng cùng test split và báo tối thiểu: COCO mask AP50-95, AP50, recall, sai số đếm mỗi ảnh, thời gian/ảnh, peak VRAM và số ảnh có hơn 200 instance. Không so `quality` SAM 3 với một YOLO run dùng split hoặc preprocessing khác.